# Exploration des prévisions CMC – HRDPS

Ce notebook permet d'explorer les données téléchargées par `cmc_fetch.py`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from cmc_fetch import download_run, load_runs
from cmc_config import MODELS, STATIONS_CONTINENTAL, STATIONS_WEST

## 1. Télécharger une seule passe (test rapide)

In [ ]:
# Dernière passe complète disponible
recent = pd.Timestamp.utcnow().floor('6h') - pd.Timedelta('6h')
print(f"Passe: {recent}")

df = download_run(
    recent,
    model_key="hrdps_continental",
    skip_if_exists=True,
)
df.head(10)

## 2. Charger toutes les données disponibles

In [ ]:
df_all = load_runs(
    model_key="hrdps_continental",
    # start="2025-03-01",
    # stids=["CYUL", "CYOW", "CYQB"],
)
print(f"Lignes: {len(df_all):,}")
print(f"Passes: {df_all['init_time'].nunique()}")
print(f"Stations: {sorted(df_all['stid'].unique())}")
df_all.head()

## 3. Prévisions de vent à un aéroport

In [ ]:
station = "CYUL"   # Montréal Trudeau
# Dernière passe disponible pour cette station
last_run = df_all['init_time'].max()

sub = df_all[
    (df_all['stid'] == station) &
    (df_all['init_time'] == last_run)
].sort_values('fxx')

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
fig.suptitle(f"{station} – Passe {last_run.strftime('%Y-%m-%d %HZ')}", fontsize=13)

if 'WIND_10m' in sub.columns:
    axes[0].plot(sub['valid_time'], sub['WIND_10m'], label='Vitesse (m/s)')
    if 'GUST_10m' in sub.columns:
        axes[0].fill_between(sub['valid_time'], sub['WIND_10m'], sub['GUST_10m'],
                             alpha=0.3, label='Rafales')
    axes[0].set_ylabel('Vent (m/s)')
    axes[0].legend()

if 'TMP_2m' in sub.columns:
    axes[1].plot(sub['valid_time'], sub['TMP_2m'] - 273.15, color='red')
    axes[1].set_ylabel('Temp (°C)')

if 'VIS_Sfc' in sub.columns:
    axes[2].plot(sub['valid_time'], sub['VIS_Sfc'] / 1000, color='purple')
    axes[2].axhline(1.6, color='gray', linestyle='--', label='IFR (1.6 km)')
    axes[2].set_ylabel('Visibilité (km)')
    axes[2].legend()

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %H:%M'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Comparaison multi-passes (spaghetti) – Vent à CYUL

In [ ]:
station = "CYUL"
# Prendre les 8 dernières passes (2 jours)
all_runs = sorted(df_all['init_time'].unique())[-8:]

fig, ax = plt.subplots(figsize=(13, 5))
for run in all_runs:
    sub = df_all[(df_all['stid'] == station) & (df_all['init_time'] == run)].sort_values('fxx')
    if 'WIND_10m' in sub.columns and len(sub) > 0:
        ax.plot(sub['valid_time'], sub['WIND_10m'],
                label=run.strftime('%d/%m %HZ'), alpha=0.7)

ax.set_title(f"{station} – Vent 10m (m/s) – Spaghetti des passes")
ax.set_ylabel('Vent (m/s)')
ax.legend(fontsize=8, ncol=4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %HZ'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 5. Tableau synthèse – Toutes les stations, 24h de prévision

In [ ]:
last_run = df_all['init_time'].max()
horizon = 24  # heures

summary = (
    df_all[
        (df_all['init_time'] == last_run) &
        (df_all['fxx'] <= horizon)
    ]
    .groupby('stid')
    .agg(
        WIND_max=('WIND_10m', 'max'),
        GUST_max=('GUST_10m', 'max'),
        TMP_min=('TMP_2m', lambda x: x.min() - 273.15),
        TMP_max=('TMP_2m', lambda x: x.max() - 273.15),
        PRATE_sum=('PRATE_Sfc', 'sum'),
        VIS_min=('VIS_Sfc', lambda x: x.min() / 1000),
    )
    .round(1)
)
summary.columns = ['Vent max (m/s)', 'Rafale max (m/s)', 'Temp min (°C)',
                   'Temp max (°C)', 'PRATE cumulée', 'Visib. min (km)']
print(f"Passe: {last_run}  |  Horizon: {horizon}h")
summary

## 6. HRDPS-West (Ouest du Canada)

In [ ]:
# Télécharger la passe la plus récente pour l'Ouest
df_west = download_run(
    recent,
    model_key="hrdps_west",
    skip_if_exists=True,
)
if df_west is not None:
    df_west.head(10)